<a href="https://colab.research.google.com/github/Snehamn24/Traditional_ML_NLP_Triage_Severity_prediction/blob/main/Day_1_ML_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Day 1 Project

## Downloading the data set

In [2]:
# download synthetic medical triage Priority dataset from kaggle into
#content/data/raw

#step 1 : install Kaggle
!pip install -q kaggle


In [4]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"snehanagarajm","key":"e78c6c7d904acf97887946f31197c211"}'}

In [5]:
# step 3 configure Kaggle API

In [6]:
import os

# Create kaggle folder
os.makedirs('/root/.kaggle', exist_ok=True)

# Move kaggle.json
!cp kaggle.json /root/.kaggle/

# Set permissions
!chmod 600 /root/.kaggle/kaggle.json

In [7]:
os.makedirs('/content/data/raw', exist_ok=True)

In [8]:
!kaggle datasets download -d azam897/synapse-dataset-of-symptoms-and-demographics -p /content/data/raw

Dataset URL: https://www.kaggle.com/datasets/azam897/synapse-dataset-of-symptoms-and-demographics
License(s): apache-2.0
100% 627k/627k [00:00<00:00, 105MB/s]



In [9]:
import zipfile

zip_path = "/content/data/raw/synapse-dataset-of-symptoms-and-demographics.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/data/raw')

In [10]:
import os
os.listdir('/content/data/raw')

['synapse-dataset-of-symptoms-and-demographics.zip',
 'SYNAPSE_An Expert Annotated Dataset of Patient symptoms and Demographics.csv']

Read the dataset

In [11]:
import pandas as pd

df = pd.read_csv('/content/data/raw/SYNAPSE_An Expert Annotated Dataset of Patient symptoms and Demographics.csv')

In [12]:
df.head()

,Symptoms,Gender,Age,Duration,Severity,Final Recommendation
0,"Unwanted weight loss, Mouth sore, Persistent m...",Male,6-15 years,Greater than 3 days,Severe,Doctor Consultation
1,"Feeling tired, Shortness breath, Blue skin col...",Female,below 5 years,Greater than 3 days,Severe,Doctor Consultation
2,"False beliefs, Seeing, hearing things are not ...",Male,6-15 years,Greater than 3 days,Mild,Doctor Consultation
3,"Chest pain, Shortness breath, Fast heart rate,...",Female,above 45 years,Greater than 3 days,Moderate,Doctor Consultation
4,"fever, vaginal bleeding, painful urination, di...",Female,below 5 years,Less than 3 days,Severe,Doctor Consultation


In [16]:
# dataset shape
r,c=df.shape
print("Rows : " ,r)
print("Columns : " , c)

Rows :  130637
Columns :  6


# Module 3 - Inspect Dataset

Before tarining , check
Required Colums exist

1.   Check for missing values
2.   All the attributes are exist



In [17]:
#checking the columns
df.columns

Index(['Symptoms', 'Gender', 'Age', 'Duration', 'Severity',
       'Final Recommendation'],
      dtype='object')

In [18]:
df.duplicated()

,0
0,False
1,False
2,False
3,False
4,False
...,...
130632,False
130633,False
130634,False
130635,False


In [19]:
df.duplicated().sum()

np.int64(0)

In [20]:
df.isnull().sum()

,0
Symptoms,0
Gender,0
Age,0
Duration,0
Severity,0
Final Recommendation,0


Module 5 - Prepare data




1.   Extract the first 80% records from the dataset that you downloaded in the   previous step . Save the extra "full_data.csv" in the folder "train"
2.  Extract the last 20% records from the daataset . Remove the column named "Severity" from the extracted records as a  file named "test_Data.csv" in the folder "test"



In [21]:
# Create folders
os.makedirs('/content/data/train', exist_ok=True)
os.makedirs('/content/data/test', exist_ok=True)

# First 10000 rows -> training data
train_df = df.iloc[:10000]

# Last 2000 rows -> testing data
test_df = df.iloc[-2000:]

# Remove Severity column from test data
test_df = test_df.drop(columns=['Severity'])

# Save files
train_df.to_csv('/content/data/train/full_data.csv', index=False)
test_df.to_csv('/content/data/test/test_Data.csv', index=False)

print("Files saved successfully!")

Files saved successfully!


Module- 5 : Clean Symptoms Text in Raw and Test


1.   Convert text to lowercase
2.   Remove unusual punctuation
3.   Normalize extra spaces
4.   Preserve medically meaningful words



In [22]:
import re

# Load datasets
train_df = pd.read_csv('/content/data/train/full_data.csv')
test_df = pd.read_csv('/content/data/test/test_Data.csv')

# Function to clean text
def clean_text(text):

    # Convert to string
    text = str(text)

    # Convert to lowercase
    text = text.lower()

    # Remove unusual punctuation
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)

    # Normalize extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply cleaning on Symptoms column
train_df['Symptoms'] = train_df['Symptoms'].apply(clean_text)
test_df['Symptoms'] = test_df['Symptoms'].apply(clean_text)

# Save cleaned files
train_df.to_csv('/content/data/train/cleaned_full_data.csv', index=False)
test_df.to_csv('/content/data/test/cleaned_test_Data.csv', index=False)

print("Cleaning completed successfully!")

Cleaning completed successfully!


Module 6 : Input features

The model learns


In [25]:
TEXT_COL = "Symptoms"
CATEGORICAL_COLS = ["Gender","Age","Duration"]
TARGET_COL = "Severity"

x = train_df[[TEXT_COL]+CATEGORICAL_COLS]
y = train_df[TARGET_COL]

print("Feature columns : " , x.columns.tolist())
print("Target column : " , TARGET_COL)
print("X shape " , x.shape)
print("Y shape : ",y.shape)

Feature columns :  ['Symptoms', 'Gender', 'Age', 'Duration']
Target column :  Severity
X shape  (10000, 4)
Y shape :  (10000,)


In [24]:
train_df.columns

Index(['Symptoms', 'Gender', 'Age', 'Duration', 'Severity',
       'Final Recommendation'],
      dtype='object')

In [28]:
from sklearn.model_selection import train_test_split
min_class_count = y.value_counts().min()

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.25,random_state=42,stratify=y)

print("Using startified train/test split")

print("Train shape : ",x_train.shape)
print("Test Shape : ",x_test.shape)

Using startified train/test split
Train shape :  (7500, 4)
Test Shape :  (2500, 4)
